# 04 — Limpieza de datos

Pandas tiene funciones específicas para cada tipo de problema en los datos. El orden correcto es siempre: tipos → duplicados → nulos → formato.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
df  = air.copy()
print(df.shape)


(279712, 33)


## Corrección de tipos

`astype()` convierte tipos directamente. `pd.to_numeric()` y `pd.to_datetime()` son más robustos cuando hay valores que no se pueden convertir.

In [2]:
# astype() — conversión directa, falla si hay valores incompatibles
# df['precio'] = df['precio'].astype(float)  # lanza error si hay strings

# pd.to_numeric con errors='coerce' — convierte a NaN lo que no puede convertir
df['host_total_listings_count'] = pd.to_numeric(
    df['host_total_listings_count'], errors='coerce'
)

# pd.to_datetime — convierte strings de fecha a datetime64
df['host_since'] = pd.to_datetime(df['host_since'])

# Booleanos desde strings con map
for col in ['host_is_superhost', 'host_has_profile_pic',
            'host_identity_verified', 'instant_bookable']:
    df[col] = df[col].map({'t': True, 'f': False})

print(df[['host_since', 'host_is_superhost', 'host_total_listings_count']].dtypes)


host_since                   datetime64[us]
host_is_superhost                    object
host_total_listings_count           float64
dtype: object


## Duplicados

In [3]:
# duplicated() — True en la segunda ocurrencia y siguientes por defecto
# keep='first' conserva la primera, marca las demás como duplicado
# keep='last'  conserva la última
# keep=False    marca todas las ocurrencias como duplicado
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')

# Duplicados por columnas específicas
n_dup_id = df.duplicated(subset='listing_id').sum()
print(f'listing_id duplicados: {n_dup_id}')

# drop_duplicates elimina y devuelve el DataFrame sin duplicados
df = df.drop_duplicates()
print(f'Shape tras limpieza: {df.shape}')


Filas duplicadas: 0
listing_id duplicados: 0


Shape tras limpieza: (279712, 33)


## Nulos — detección

In [4]:
# isnull() / isna() — equivalentes
# notnull() / notna() — lo contrario
nulos = df.isnull().sum().sort_values(ascending=False)
pct   = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)

resumen = pd.DataFrame({'conteo': nulos, 'pct': pct})
print(resumen[resumen['conteo'] > 0])


                             conteo   pct
bedrooms                      29435  10.5
district                     242700  86.8
host_acceptance_rate         113087  40.4
host_has_profile_pic            165   0.1
host_identity_verified          165   0.1
host_is_superhost               165   0.1
host_location                   840   0.3
host_response_rate           128782  46.0
host_response_time           128782  46.0
host_since                      165   0.1
host_total_listings_count       165   0.1
name                            175   0.1
review_scores_accuracy        91713  32.8
review_scores_checkin         91771  32.8
review_scores_cleanliness     91665  32.8
review_scores_communication   91687  32.8
review_scores_location        91775  32.8
review_scores_rating          91405  32.7
review_scores_value           91785  32.8


## Nulos — fillna

In [5]:
# Rellenar con valor fijo
df['host_response_time'] = df['host_response_time'].fillna('sin_dato')

# Rellenar con estadística
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())

# Rellenar con estadística de grupo (más preciso)
df['host_response_rate'] = df['host_response_rate'].fillna(
    df.groupby('city')['host_response_rate'].transform('median')
)

# ffill — propaga el valor anterior (útil en series temporales ordenadas)
# bfill — propaga el valor siguiente
# df['col'] = df['col'].ffill()

print('Nulos bedrooms:', df['bedrooms'].isnull().sum())


Nulos bedrooms: 0


## Nulos — dropna

In [6]:
# how='any' (default) — elimina si hay al menos un nulo en la fila
# how='all' — elimina solo si todos los valores de la fila son nulos
# subset — solo mira esas columnas
# thresh — conserva filas con al menos N valores no nulos

antes = len(df)

# Eliminar filas donde 'name' es nulo
df_sin_nombre = df.dropna(subset=['name'])
print(f'Eliminadas por name nulo: {antes - len(df_sin_nombre)}')

# Conservar solo filas con al menos 25 valores no nulos (de 33 columnas)
df_completo = df.dropna(thresh=25)
print(f'Eliminadas por thresh=25: {antes - len(df_completo)}')


Eliminadas por name nulo: 175


Eliminadas por thresh=25: 48704


## replace() — corregir valores específicos

In [7]:
from pandas import read_csv
df2 = read_csv(AIR, encoding='latin-1', low_memory=False)

# Reemplazar valores concretos
df2['host_is_superhost'] = df2['host_is_superhost'].replace({'t': True, 'f': False})

# Reemplazar con regex
# df['col'] = df['col'].str.replace(r'\$|,', '', regex=True)

# Reemplazar múltiples valores con un diccionario
# df['rating'] = df['rating'].replace({'Excellent': 5, 'Good': 4, 'Poor': 1})

# Reemplazar con NaN
import numpy as np
df2['district'] = df2['district'].replace('', np.nan)
print(df2['host_is_superhost'].value_counts(dropna=False).head())


host_is_superhost
False    229294
True      50253
NaN         165
Name: count, dtype: int64


---
## Resumen

| Operación | Sintaxis |
|-----------|----------|
| Convertir tipo | `df['col'].astype(float)` |
| Convertir número (robusto) | `pd.to_numeric(df['col'], errors='coerce')` |
| Convertir fecha | `pd.to_datetime(df['col'])` |
| Detectar nulos | `df.isnull().sum()` |
| Rellenar nulos | `df['col'].fillna(valor)` |
| Rellenar por grupo | `df.groupby('g')['col'].transform('median')` |
| Eliminar filas con nulos | `df.dropna(subset=['col'])` |
| Eliminar duplicados | `df.drop_duplicates()` |
| Reemplazar valores | `df['col'].replace({'a': 'b'})` |
